本 notebook 是 [M04.4 树模型与集成学习](../notes/04-trees-ensembles.qmd) 的实例讲解。

1. 1. 从零一维 CART 与框架对齐
2. 2. Bagging 的相关性下限与袋外估计
3. 3. 平方损失提升就是拟合残差
4. 4. 直方图近似与外推边界

在 VS Code 中选择仓库 `.venv` 内核，逐单元运行；运行结果随文件保存。示例使用固定随机种子和合成数据。

In [1]:
import numpy as np
from matplotlib_inline.backend_inline import set_matplotlib_formats
from sklearn.ensemble import (
    GradientBoostingRegressor,
    HistGradientBoostingRegressor,
    RandomForestRegressor,
)
from sklearn.tree import DecisionTreeRegressor

from semiai.plotting import use_style

use_style()
set_matplotlib_formats("svg")
rng = np.random.default_rng(42)

## 1. 从零一维 CART 与框架对齐

遍历相邻观测的中点，以叶内平方残差和选择切分，递归到指定深度。

In [2]:
def grow(x, y, depth=0, max_depth=3, min_leaf=4):
    """一维合成回归树；节点存储阈值或叶均值。"""
    if depth == max_depth or len(y) < 2 * min_leaf:
        return float(y.mean())
    best = None
    cost = np.inf
    unique = np.unique(x)
    for threshold in (unique[:-1] + unique[1:]) / 2:
        left = x <= threshold
        right = ~left
        if min(left.sum(), right.sum()) < min_leaf:
            continue
        value = np.sum((y[left] - y[left].mean()) ** 2) + np.sum((y[right] - y[right].mean()) ** 2)
        if value < cost:
            cost = value
            best = (threshold, left)
    if best is None:
        return float(y.mean())
    threshold, left = best
    return (
        threshold,
        grow(x[left], y[left], depth + 1, max_depth, min_leaf),
        grow(x[~left], y[~left], depth + 1, max_depth, min_leaf),
    )


def predict_tree(tree, x):
    if not isinstance(tree, tuple):
        return np.full(len(x), tree)
    threshold, left, right = tree
    mask = x <= threshold
    out = np.empty(len(x))
    out[mask] = predict_tree(left, x[mask])
    out[~mask] = predict_tree(right, x[~mask])
    return out


x = np.sort(rng.uniform(-2, 2, 100))
y = np.sin(2 * x) + rng.normal(0, 0.2, len(x))
tree = grow(x, y)
grid = np.linspace(-2, 2, 301)
reference = DecisionTreeRegressor(max_depth=3, min_samples_leaf=4, random_state=42).fit(
    x[:, None], y
)
error = np.max(np.abs(predict_tree(tree, grid) - reference.predict(grid[:, None])))
print("根阈值:", tree[0], reference.tree_.threshold[0], "预测最大差:", error)
assert error < 1e-10

根阈值: 0.21632837534766125 0.21632837504148483 预测最大差: 1.1102230246251565e-16


读法：这里限定一维、平方损失、无缺失、无并列最佳切分；一般 CART 还要搜索特征，并处理权重、分类不纯度和剪枝。

## 2. Bagging 的相关性下限与袋外估计

先用已知相关结构验证集成方差，再运行实际森林并区分袋外预测与正式留出评估。

In [3]:
rho = 0.4
m = 50
draws = 10000
common = rng.normal(size=(draws, 1))
individual = rng.normal(size=(draws, m))
ensemble = np.sqrt(rho) * common + np.sqrt(1 - rho) * individual
print("平均器方差:", ensemble.mean(axis=1).var(), "理论:", rho + (1 - rho) / m)
X = rng.normal(size=(350, 4))
y = np.sin(X[:, 0]) + X[:, 1] ** 2 + rng.normal(0, 0.2, len(X))
model = RandomForestRegressor(
    n_estimators=120, oob_score=True, min_samples_leaf=3, max_features=0.75, random_state=42
).fit(X, y)
print("训练 RMSE:", np.sqrt(np.mean((y - model.predict(X)) ** 2)))
print("OOB RMSE:", np.sqrt(np.mean((y - model.oob_prediction_) ** 2)), "OOB R²:", model.oob_score_)

平均器方差: 0.41897813597757705 理论: 0.41200000000000003
训练 RMSE: 0.27405116311967737
OOB RMSE: 0.41319963457732417 OOB R²: 0.9247458809596624


读法：增加树数只压低独立部分，树间共同误差不会消失；OOB 使用未含该行的 bootstrap 树，但不自动满足轨迹分组或时间边界。

## 3. 平方损失提升就是拟合残差

手写逐轮残差拟合，用深度为一的树作为基学习器，与相同设置的 GradientBoostingRegressor 对齐。

In [4]:
x = np.sort(rng.uniform(-2, 2, 150))[:, None]
y = np.sin(2 * x[:, 0]) + rng.normal(0, 0.15, len(x))
eta = 0.1
rounds = 40
prediction = np.full(len(y), y.mean())
history = []
for _ in range(rounds):
    stump = DecisionTreeRegressor(max_depth=1, random_state=42).fit(x, y - prediction)
    prediction += eta * stump.predict(x)
    history.append(np.mean((y - prediction) ** 2))
reference = GradientBoostingRegressor(
    n_estimators=rounds, learning_rate=eta, max_depth=1, random_state=42
).fit(x, y)
print("手写/框架预测差:", np.max(np.abs(prediction - reference.predict(x))))
print("首末训练 MSE:", history[0], history[-1])
assert np.allclose(prediction, reference.predict(x), atol=1e-10)

手写/框架预测差: 1.6653345369377348e-16
首末训练 MSE: 0.4787630804218181 0.15162546606314015


读法：平方损失的负梯度与残差成比例；其他损失需相应伪残差和叶值更新。更多轮数降低训练误差，但正式早停应使用训练侧验证集。

## 4. 直方图近似与外推边界

实际运行 sklearn 的直方图提升，查看训练范围外的预测，连接分箱和树叶常数的机制。

In [5]:
x = rng.uniform(-1, 1, (400, 1))
y = 2 * x[:, 0] + rng.normal(0, 0.05, len(x))
for bins in [8, 32, 128]:
    model = HistGradientBoostingRegressor(
        max_bins=bins,
        max_iter=60,
        max_leaf_nodes=7,
        min_samples_leaf=10,
        early_stopping=False,
        random_state=42,
    ).fit(x, y)
    probes = np.array([-3.0, -1.5, 1.5, 3.0])[:, None]
    print(
        "bins",
        bins,
        "训练 MSE",
        np.mean((model.predict(x) - y) ** 2),
        "范围外预测",
        model.predict(probes),
    )

bins 8 训练 MSE 0.02394909348170879 范围外预测 [-1.71220111 -1.71220111  1.74613526  1.74613526]


bins 32 训练 MSE 0.004169055221448952 范围外预测 [-1.9216364  -1.9216364   1.95481416  1.95481416]


bins 128 训练 MSE 0.0021913290888344182 范围外预测 [-1.94858521 -1.94858521  1.95385276  1.95385276]


读法：分箱减少候选阈值与内存，并可能损失细节；常数叶树在最外侧区域保持常数，不能自然延续真实线性趋势。这里不是 XGBoost/LightGBM 的速度测评。

## 小结

CART 用局部切分降低损失，Bagging 用平均减少不稳定性，森林用随机化降低相关，Boosting 用逐轮梯度修正函数。工程优化改变搜索和存储成本，不能替代评估协议，也不改变模型对输入覆盖与目标光滑性的基本限制。